In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ─── Hyperparameters ───
SEQ_LEN       = 12    # input timesteps (1 hour at 5-min intervals)
PRED_LEN      = 12    # forecast horizon (1 hour ahead)
BATCH_SIZE    = 64
HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
DROPOUT       = 0.1
LEARNING_RATE = 1e-3
NUM_EPOCHS    = 30
TRAIN_RATIO   = 0.7
VAL_RATIO     = 0.1
# TEST_RATIO  = 0.2  (remainder)

In [ ]:
DATA_DIR = './data'

def load_pems(name, path):
    raw = np.load(path, allow_pickle=True)
    data = raw['data']  # (T, N, F)
    print(f'{name}: shape={data.shape}')
    return data[:, :, 0]  # use flow only → (T, N)

datasets = {}
pems_paths = {
    'PeMS04': os.path.join(DATA_DIR, 'PEMS04/pems04.npz'),
    'PeMS07': os.path.join(DATA_DIR, 'PEMS07/pems07.npz'),
    'PeMS08': os.path.join(DATA_DIR, 'PEMS08/pems08.npz'),
}
for name, path in pems_paths.items():
    try:
        datasets[name] = load_pems(name, path)
    except FileNotFoundError:
        print(f'[SKIP] {name} not found.')

# NYCTaxi — adapt column names to your file
try:
    nyc_df = pd.read_csv(os.path.join(DATA_DIR, 'NYCTaxi/nyc_taxi.csv'))
    numeric_cols = nyc_df.select_dtypes(include=[np.number]).columns.tolist()
    datasets['NYCTaxi'] = nyc_df[numeric_cols].values  # (T, N)
    print(f'NYCTaxi: shape={datasets["NYCTaxi"].shape}')
except FileNotFoundError:
    print('[SKIP] NYCTaxi not found.')